# Apparent Temperature Estimation

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data"
OUT = ROOT / "artifacts"
OUT.mkdir(exist_ok=True)
keras.utils.set_random_seed(42)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
frame = pd.read_csv(DATA / "weatherHistory.csv")
frame["Formatted Date"] = pd.to_datetime(frame["Formatted Date"], utc=True)
frame = frame.dropna(subset=["Formatted Date", "Apparent Temperature (C)"]).sort_values("Formatted Date").drop_duplicates("Formatted Date").reset_index(drop=True)
num = ["Temperature (C)", "Humidity", "Wind Speed (km/h)", "Pressure (millibars)"]
cat = ["Precip Type"]
frame.loc[frame["Pressure (millibars)"] <= 0, "Pressure (millibars)"] = np.nan
cut = int(len(frame) * 0.8)
train, test = frame.iloc[:cut], frame.iloc[cut:]
if len(train) < 2 or test.empty:
    raise ValueError("Dataset is too small for a chronological evaluation.")
assert train["Formatted Date"].max() < test["Formatted Date"].min()
x_train, x_test = train[num + cat], test[num + cat]
y_train, y_test = train["Apparent Temperature (C)"], test["Apparent Temperature (C)"]
def preprocessing():
    return ColumnTransformer([("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num), ("categorical", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False)), cat)])

In [ ]:
models = {"ridge": make_pipeline(preprocessing(), Ridge()), "gradient_boosting": make_pipeline(preprocessing(), HistGradientBoostingRegressor(random_state=42, early_stopping=False))}
predictions = {"train_median": np.full(len(test), y_train.median())}
for name, model in models.items():
    model.fit(x_train, y_train)
    predictions[name] = model.predict(x_test)
results = {name: {"mae_celsius": mean_absolute_error(y_test, pred), "rmse_celsius": float(np.sqrt(mean_squared_error(y_test, pred)))} for name, pred in predictions.items()}
print(json.dumps(results, indent=2))
(OUT / "metrics.json").write_text(json.dumps(results, indent=2))

In [ ]:
import matplotlib.pyplot as plt
import joblib
for name, model in models.items():
    joblib.dump(model, OUT / f"{name}.joblib")
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, predictions["gradient_boosting"], s=5, alpha=0.25)
lo, hi = y_test.min(), y_test.max()
ax.plot([lo, hi], [lo, hi], "--", color="black")
ax.set(xlabel="Observed apparent temperature (°C)", ylabel="Estimated apparent temperature (°C)", title="Chronological test partition")
fig.tight_layout()
fig.savefig(OUT / "predicted_vs_observed.png")